# Train the Stepping Model on Kaggle

This notebook downloads the weather data, trains the stepping model on a free Kaggle GPU and scores it at every lead time out to 10 days.

Before running, check two settings in the panel on the right, under Session options:

- Accelerator: GPU T4 x2
- Internet: On

Then choose Save Version, pick Save & Run All (Commit) and save. The run happens on Kaggle's machines, so the tab can be closed.

`MODE = "test"` takes about 10 minutes and proves that everything works. `MODE = "full"` is the real training run and takes several hours. `MODE = "no-rollout"` trains the same network for as long on single steps only, to measure what rollout training adds, and takes about as long as the full run.

When the run has finished, the scores are in `stepper/results.json` on the notebook's Output tab.

In [ ]:
MODE = "test"  # "test" is a 10-minute check, "full" the real run, "no-rollout" the comparison

import os
import shutil
import subprocess
import sys
import time

import torch

assert MODE in ("test", "full", "no-rollout"), 'MODE must be "test", "full" or "no-rollout"'
assert torch.cuda.is_available(), (
    "No GPU found. In the right-hand panel set Accelerator to GPU T4 x2, then run again."
)
print(
    torch.cuda.get_device_name(0), "| torch", torch.__version__, "| python", sys.version.split()[0]
)

In [ ]:
def free_gb(path):
    return shutil.disk_usage(path).free / 2**30


# the code and the downloaded data go where Kaggle does not save them with the notebook
SCRATCH = "/tmp"
try:
    os.makedirs("/kaggle/temp", exist_ok=True)
    if free_gb("/kaggle/temp") > free_gb("/tmp"):
        SCRATCH = "/kaggle/temp"
except OSError:
    pass
# only what is written here is kept: the scores, the training history and the weights
OUT = "/kaggle/working/stepper" if os.path.isdir("/kaggle/working") else os.path.abspath("stepper")
print(f"scratch space {SCRATCH} ({free_gb(SCRATCH):.0f} GB free), results in {OUT}")

REPO = f"{SCRATCH}/weather-forecaster"
URL = "https://github.com/KairavT/weather-forecaster.git"
if not os.path.isdir(REPO):
    # the code lives on the no-rollout branch until its pull request is merged
    for branch in (["--branch", "no-rollout"], []):
        cloned = subprocess.run(["git", "clone", "-q", *branch, URL, REPO], check=False)
        if cloned.returncode == 0:
            break
    else:
        raise RuntimeError(
            "Could not download the code. In the right-hand panel turn Internet on, then run again."
        )
os.chdir(REPO)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "zarr>=3", "gcsfs", "certifi"], check=True
)
os.environ["PYTHONPATH"] = f"{REPO}/src"
commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True
)
print("code at commit", commit.stdout.strip())


def run(*args):
    """Run one of the repo's scripts, show its output live, and stop the notebook if it fails."""
    process = subprocess.Popen(
        [sys.executable, "-u", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{' '.join(args)} failed, see the output above")

Download 23 weather fields and two fixed maps. The test needs four years (about 1 GB), the other two runs need 1979 to 2018 (about 11 GB). Anything already downloaded is skipped.

In [ ]:
YEARS = "test" if MODE == "test" else "all"
START, END = {
    "test": ("2014-01-01T00", "2017-12-31T18"),
    "all": ("1979-01-01T00", "2018-12-31T18"),
}[YEARS]
DATA = f"{SCRATCH}/era5_{YEARS}"
started = time.time()
run("scripts/download.py", "--preset", "forecast", "--start", START, "--end", END, "--out", DATA)
print(f"download took {time.time() - started:.0f} seconds")

Train on single 6-hour steps first, then on rollouts of 2, 4 and 8 steps, and score the result on the test years. The no-rollout run trains on single steps for all 32 epochs instead, with the same number of updates and network steps, so the only difference between the two runs is the rollouts. The long runs stop training early if it gets close to Kaggle's session limit, so it always ends with a scored model.

In [ ]:
args = [
    "scripts/train_stepper.py",
    "--data",
    DATA,
    "--runs",
    f"{OUT}/runs",
    "--results",
    f"{OUT}/results.json",
]
args += {
    "test": ["--quick"],
    "full": ["--max-hours", "7"],
    "no-rollout": ["--max-hours", "7", "--phases", "1:32", "--name", "stepper-no-rollout"],
}[MODE]
run(*args)

A summary of the run. The same numbers, for every field and every lead time, are in `stepper/results.json`.

In [ ]:
import json

with open(f"{OUT}/results.json") as f:
    results = json.load(f)

for name, entry in results.items():
    print(
        f"{name}: {entry['device']}, {entry['parameters']:,} parameters, {entry['epochs_run']} epochs "
        f"in {entry['train_seconds']} seconds, peak GPU memory {entry['peak_gpu_gb']} GB"
    )
    headline = {"Z500": "geopotential_500", "T850": "temperature_850", "T2M": "2m_temperature"}
    for short, channel in headline.items():
        daily = entry["test_rmse"][channel][3::4]  # every fourth 6-hour step is a whole day
        print(f"  {short} error after 1 to {len(daily)} days:", " ".join(f"{v:.4g}" for v in daily))